In [1]:
# ═══════════════════════════════════════════════════════════════════════════
#  FAST OPTUNA CALIBRATION (REPLACES GRID SEARCH)
# ═══════════════════════════════════════════════════════════════════════════

import optuna
from joblib import Parallel, delayed


# ──────────────────────────────────────────────────────────────────────────
# PRECOMPUTE FEATURES (ONE-TIME)
# ──────────────────────────────────────────────────────────────────────────
def precompute_for_calibration(all_data, tickers):
    precomp = {}

    for ticker in tickers:
        returns = all_data[ticker]
        n = len(returns)
        train_n = int(n * TRAIN_FRAC)

        H_mid, d_alpha, rho_min, _ = extract_mf_risk_features(returns)
        ewma_vol = _compute_ewma_vol(returns)

        precomp[ticker] = {
            "returns": returns,
            "train_n": train_n,
            "H_mid": H_mid,
            "d_alpha": d_alpha,
            "rho_min": rho_min,
            "ewma_vol": ewma_vol
        }

    return precomp


# ──────────────────────────────────────────────────────────────────────────
# FAST VECTORIZED MF-VAR
# ──────────────────────────────────────────────────────────────────────────
def fast_mf_var(pre, alpha, gamma, beta, delta):
    r = pre["returns"]

    H   = np.clip(pre["H_mid"], 0.1, 0.9)
    da  = np.clip(pre["d_alpha"], 0., 2.)
    rho = np.clip(pre["rho_min"], -1., 1.)
    ewma = pre["ewma_vol"]

    z = stats.norm.ppf(alpha)

    mu = pd.Series(r).rolling(PARAM_VAR_WINDOW).mean().values
    sig = pd.Series(r).rolling(PARAM_VAR_WINDOW).std().values + 1e-10

    base = -(mu + z * sig)

    mf_raw = base * (H / 0.5)**gamma \
                  * (1 + beta * da) \
                  * (1 + delta * (1 - rho))

    param = base

    lo, hi = MF_BLEND_LOW_VOL, MF_BLEND_HIGH_VOL
    w = np.clip((ewma - lo) / (hi - lo), 0., 1.)

    return np.where(np.isfinite(mf_raw), w * mf_raw + (1 - w) * param, np.nan)


# ──────────────────────────────────────────────────────────────────────────
# LOSS FUNCTION (PER TICKER)
# ──────────────────────────────────────────────────────────────────────────
def eval_ticker_fast(pre, params_99, params_95):
    r = pre["returns"]
    train_n = pre["train_n"]
    r_test = r[train_n:]

    losses = []

    for alpha, p in [(0.01, params_99), (0.05, params_95)]:
        var = fast_mf_var(pre, alpha, p["gamma"], p["beta"], p["delta"])
        v = var[train_n:]

        mask = np.isfinite(v)
        if mask.sum() < 100:
            continue

        r_t = r_test[mask]
        v_t = v[mask]

        breaches = (r_t < -v_t)
        breach_rate = breaches.mean()

        # Calibration error
        cal_err = abs(breach_rate - alpha)

        # Kupiec penalty
        _, p_val, _ = kupiec_pof(breaches, len(r_t), alpha)
        kup_penalty = max(0, 0.05 - (p_val if np.isfinite(p_val) else 0))

        # Autocorrelation penalty
        b = breaches.astype(float)
        ac = np.corrcoef(b[:-1], b[1:])[0,1] if len(b) > 10 and b.std() > 1e-10 else 0

        loss = cal_err + 0.5 * kup_penalty + 0.2 * abs(ac)
        losses.append(loss)

    return np.mean(losses) if losses else 1.0


# ──────────────────────────────────────────────────────────────────────────
# MAIN OPTUNA CALIBRATION
# ──────────────────────────────────────────────────────────────────────────
def calibrate_mf_optuna(all_data, n_tickers=10, n_trials=60):
    tickers = [t for t in ALL_TICKERS if t in all_data][:n_tickers]

    print(f"[Optuna] Precomputing features for {len(tickers)} tickers...")
    precomp = precompute_for_calibration(all_data, tickers)

    def objective(trial):
        # Separate params for 99% and 95%
        params_99 = {
            "gamma": trial.suggest_float("g99", 0.3, 0.7),
            "beta":  trial.suggest_float("b99", 0.1, 0.4),
            "delta": trial.suggest_float("d99", 0.1, 0.3),
        }
        params_95 = {
            "gamma": trial.suggest_float("g95", 0.05, 0.2),
            "beta":  trial.suggest_float("b95", 0.05, 0.2),
            "delta": trial.suggest_float("d95", 0.03, 0.1),
        }

        losses = Parallel(n_jobs=-1)(
            delayed(eval_ticker_fast)(precomp[t], params_99, params_95)
            for t in tickers
        )

        return float(np.mean(losses))

    print(f"[Optuna] Running {n_trials} trials...")
    study = optuna.create_study(direction="minimize")
    study.optimize(objective, n_trials=n_trials, show_progress_bar=True)

    best = study.best_params

    print("\n[Optuna] BEST PARAMS FOUND:")
    print(best)

    return {
        0.01: {"gamma": best["g99"], "beta": best["b99"], "delta": best["d99"]},
        0.05: {"gamma": best["g95"], "beta": best["b95"], "delta": best["d95"]},
    }

In [7]:
"""
MF Risk Benchmark — v5 FINAL
=============================

KEY CHANGES FROM ORIGINAL v5
------------------------------

1. CONTINUOUS dk CORRECTION (replaces binary gate)
   ─────────────────────────────────────────────────
   OLD (binary):
       if alpha != 0.01 or dk_signal < DK_THRESHOLD:
           return base          # ← silently zero, no explanation
       correction = 1 + ETA * dk_signal

   NEW (continuous sigmoid):
       dk_weight  = sigmoid((dk_estimated - DK_PIVOT) / DK_SCALE)
       dk_applied = dk_estimated * dk_weight   # smooth 0→full
       correction = 1 + ETA * dk_applied       # at ALL alphas, scaled

   Rationale:
     - Binary gates create discontinuities in VaR surfaces
     - A signal of 0.049 vs 0.051 should not flip the correction 0/+25%
     - The sigmoid provides proportional, differentiable scaling
     - Alpha-sensitivity preserved via ETA_BY_ALPHA dict (larger at 0.01)

2. FULL PER-TICKER DIAGNOSTICS (7 sections)
   ──────────────────────────────────────────
   Sections printed for every ticker:
     1. Non-constancy diagnostic  (flagged horizons, rel/abs/spearman/coverage)
     2. D_k structure table       (rows=k, cols=t, mean, range, ΔD_k)
     3. Practical significance    (negligible / moderate / large vs thresholds)
     4. VaR performance context   (breach rates all 5 models)
     5. dk activation decision    (ALWAYS prints dk_estimated AND dk_applied)
     6. Counterfactual            (forced-ON impact)
     7. Interpretation            (structural non-Gaussianity vs model misspec)

3. NEVER SUPPRESS dk SILENTLY
   ────────────────────────────
   dk_estimated: raw computed signal (always printed)
   dk_applied:   sigmoid-weighted version (always printed)
   These are logged per ticker in all_results["dk_estimated"] / ["dk_applied"].

EVALUATION METRICS (unchanged from v5)
   Kupiec POF | Christoffersen CC | ES ratio | Capital efficiency | Breach AC
"""

import warnings, os, time, logging
warnings.filterwarnings("ignore")
logging.getLogger("optuna").setLevel(logging.WARNING)

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from scipy import stats
from scipy.signal import savgol_filter
from scipy.stats import spearmanr
import yfinance as yf

try:
    from arch import arch_model
    HAS_ARCH = True
except ImportError:
    HAS_ARCH = False
    print("[WARN] arch not found — using EWMA fallback. pip install arch")

import optuna
from optuna.samplers import TPESampler
from optuna.pruners import MedianPruner
from joblib import Parallel, delayed

try:
    import cupy as cp
    _gpu_mem = cp.cuda.Device(0).mem_info
    GPU_AVAILABLE = True
    print(f"[GPU] CuPy detected — {_gpu_mem[1]/2**30:.1f} GB total, "
          f"{_gpu_mem[0]/2**30:.1f} GB free")
except Exception as _e:
    cp = None
    GPU_AVAILABLE = False
    print(f"[GPU] CuPy not available ({_e}) — using NumPy")

# ─── Universe ─────────────────────────────────────────────────────────────────
SECTORS = {
    "Tech":                  ["AAPL","MSFT","GOOG","AMZN","NVDA","META","TSLA","NFLX","AMD","INTC"],
    "Financials":            ["JPM","BAC","GS","MS","WFC","BLK","C","AXP","V","MA"],
    "Healthcare & Consumer": ["JNJ","PFE","UNH","MRK","ABBV","PG","KO","PEP","WMT","COST"],
    "Energy & Industrials":  ["XOM","CVX","COP","BA","CAT","GE","HON","LMT","RTX","UPS"],
    "Commodities & Crypto":  ["GC=F","SI=F","CL=F","BTC-USD","ETH-USD"],
    "Intl ETFs":             ["EWJ","EWZ","FXI","EWG","EWU"],
}
ALL_TICKERS = [t for ts in SECTORS.values() for t in ts]
SECTOR_OF   = {t: s for s, ts in SECTORS.items() for t in ts}

START, END   = "2000-01-01", "2026-04-11"
TRAIN_FRAC   = 0.75

# ─── MF-DFA parameters ────────────────────────────────────────────────────────
MF_WINDOW  = 120
MF_STEP    = 10
Q_VALS     = np.array([-10,-5,-2,-1,0,1,2,5,10], dtype=float)

# ─── s_k / D_k parameters (paper Section 5.5, Eq. 19) ────────────────────────
SK_N_ORIGINS              = 500
SK_HORIZONS               = [1, 5, 10, 20, 40, 60, 120, 250]
SK_K_VALUES               = [0.5, 1.0, 2.0, 3.0, 3.5, 4.0, 5.0, 6.0, 7.0, 8.0, 9.0, 10.0]
SK_CF_MIN_MAG             = 0.40
SK_MIN_VALID_K            = 4
SK_RELATIVE_CHANGE_THRESH = 0.10   # δ* = 10 % (Eq. 21)
SK_ABSOLUTE_CHANGE_THRESH = 5e-5
SK_VALIDITY_RATIO         = 0.60
SK_MIN_ABS_MEAN           = 1e-5

# ─── VaR parameters ───────────────────────────────────────────────────────────
HIST_VAR_WINDOW  = 250
PARAM_VAR_WINDOW = 60
ALPHAS           = [0.01, 0.05]

# ─── MF-VaR scaling constants (Optuna-calibrated per alpha) ───────────────────
MF_PARAMS = {
    0.01: {"gamma": 0.50, "beta": 0.30, "delta": 0.20},
    0.05: {"gamma": 0.15, "beta": 0.10, "delta": 0.08},
}

# ─── GARCH-hybrid blend thresholds ────────────────────────────────────────────
EWMA_LAMBDA       = 0.94
MF_BLEND_LOW_VOL  = 0.008
MF_BLEND_HIGH_VOL = 0.020

# ─── CONTINUOUS dk CORRECTION PARAMETERS ──────────────────────────────────────
#
#  NEW DESIGN: dk is a continuous, sigmoid-weighted correction applied at
#  ALL alpha levels (not just 0.01), scaled proportionally to the estimated
#  non-constancy signal.
#
#  dk_weight(x)  = 1 / (1 + exp(-(x - DK_PIVOT) / DK_SCALE))
#  dk_applied    = dk_estimated * dk_weight(dk_estimated)
#  correction    = 1 + ETA_BY_ALPHA[alpha] * dk_applied
#
#  DK_PIVOT : signal level at which sigmoid = 0.5 (half weight applied)
#  DK_SCALE : sigmoid steepness — smaller = sharper transition
#  ETA_BY_ALPHA: magnitude of the correction; larger at 0.01 (tail matters more)
#
#  Compared to the old binary gate:
#    Signal = 0.00 → weight ≈ 0.006 → correction ≈ +0.0%  (functionally zero)
#    Signal = 0.05 → weight ≈ 0.269 → correction ≈ +1-3%  (gentle ramp)
#    Signal = 0.10 → weight ≈ 0.500 → correction ≈ +6-13% (half correction)
#    Signal = 0.20 → weight ≈ 0.731 → correction ≈ +18%   (near full)
#    Signal = 0.50 → weight ≈ 0.924 → correction ≈ +23%   (saturating)
#    Signal = 1.00 → weight ≈ 0.982 → correction ≈ +25%   (full correction)
#
DK_PIVOT     = 0.10   # sigmoid midpoint — 10% rel_change (= δ* threshold)
DK_SCALE     = 0.05   # sigmoid steepness
ETA_BY_ALPHA = {
    0.01: 0.25,        # tail-sensitive: larger correction
    0.05: 0.12,        # body of distribution: smaller correction
}

# ─── Breach shock scaling ──────────────────────────────────────────────────────
MF_SHOCK_SCALE = 0.40

# ─── Optuna calibration settings ──────────────────────────────────────────────
CALIB_N_TICKERS  = 20
CALIB_N_TRIALS   = 150
CALIB_BATCH_SIZE = 8
_BOUNDS   = {
    "g99": (0.20, 0.80), "b99": (0.05, 0.60), "d99": (0.05, 0.40),
    "g95": (0.02, 0.30), "b95": (0.02, 0.30), "d95": (0.01, 0.20),
}
_PRIOR_99 = {"g99": 0.50, "b99": 0.30, "d99": 0.20}
_PRIOR_95 = {"g95": 0.15, "b95": 0.10, "d95": 0.08}
W_CAL = 1.0; W_KUP = 0.5; W_AC = 0.3

OUT_DIR = "mf_risk_outputs_v5_final"
os.makedirs(OUT_DIR, exist_ok=True)

DARK = {
    "fig":"#0d1117","ax":"#161b22","spine":"#30363d",
    "text":"#e6edf3","muted":"#8b949e",
    "blue":"#58a6ff","green":"#3fb950","red":"#f85149",
    "amber":"#e3b341","purple":"#bc8cff","teal":"#17becf",
    "orange":"#f0883e",
}
MODEL_COLORS = {
    "Historical": DARK["blue"],
    "Parametric":  DARK["teal"],
    "GARCH":       DARK["amber"],
    "MF-VaR":      DARK["green"],
    "MF-VaR-Dk":   DARK["purple"],
}


# ═══════════════════════════════════════════════════════════════════════════════
#  DIAGNOSTIC PRINTER  (7 sections)
# ═══════════════════════════════════════════════════════════════════════════════

def _div(char="=", n=64): return char * n

def print_diagnostic(ticker, returns, dk_estimated, dk_applied,
                     flagged, horizon_dk, sk_raw_results,
                     eval_by_alpha, H_mid_mean, d_alpha_mean, rho_min_mean):
    """
    Full 7-section per-ticker diagnostic.

    Parameters
    ----------
    ticker         : str
    returns        : np.ndarray  — full log-return series
    dk_estimated   : float       — raw signal before sigmoid weighting
    dk_applied     : float       — sigmoid-weighted signal (used in VaR)
    flagged        : list        — flag dicts from _detect_nonconstancy
    horizon_dk     : dict        — {t: rel_change or 0}
    sk_raw_results : dict        — {k: {t: (sk, d_k, sk_over_k2, n, mag)}}
    eval_by_alpha  : dict        — {alpha: {model: metrics_dict}}
    H_mid_mean     : float
    d_alpha_mean   : float
    rho_min_mean   : float
    """
    sep  = _div("=")
    sep2 = _div("-")
    print(f"\n{sep}")
    print(f"  [{ticker}]")
    print(f"{sep}")

    # ── 1. NON-CONSTANCY DIAGNOSTIC ──────────────────────────────────────────
    print(f"\n{'1. NON-CONSTANCY DIAGNOSTIC':}")
    print(sep2)
    if not flagged:
        print("  No horizons flagged as non-constant.")
        print(f"  → All {len(SK_HORIZONS)} horizons pass D_k constancy check.")
        verdict_str = "WEAK"
        verdict_label = "Weak / no non-constancy"
    else:
        print(f"  Flagged horizons: {len(flagged)} / {len(SK_HORIZONS)}")
        print()
        print(f"  {'t':>5}  {'rel_change':>12}  {'abs_change':>12}  "
              f"{'Spearman_rho':>13}  {'Spearman_p':>11}  {'coverage':>9}  {'direction':>10}")
        print(f"  {'-'*5}  {'-'*12}  {'-'*12}  {'-'*13}  {'-'*11}  {'-'*9}  {'-'*10}")
        all_rels = []
        for f in sorted(flagged, key=lambda x: x["t"]):
            rel = f["rel_change"] if not np.isnan(f["rel_change"]) else f["abs_change"]/SK_ABSOLUTE_CHANGE_THRESH
            all_rels.append(rel)
            print(f"  {f['t']:>5}  {rel:>12.4f}  {f['abs_change']:>12.6f}  "
                  f"{f['spearman_r']:>13.4f}  {f['spearman_p']:>11.4f}  "
                  f"{f['coverage']:>9.2f}  {f['direction']:>10}")
        mean_rel = np.mean(all_rels) if all_rels else 0.0
        max_rel  = np.max(all_rels)  if all_rels else 0.0
        print(f"\n  Mean rel_change across flagged: {mean_rel:.4f}")
        print(f"  Max  rel_change across flagged: {max_rel:.4f}")
        short_flagged = [f for f in flagged if f["t"] <= 20]
        print(f"  Short-horizon flags (t≤20):     {len(short_flagged)} / {sum(1 for t in SK_HORIZONS if t<=20)}")
        if max_rel > 0.50 or len(flagged) >= 4:
            verdict_str   = "STRONG"
            verdict_label = "Strong non-constancy"
        elif max_rel > 0.20 or len(flagged) >= 2:
            verdict_str   = "MODERATE"
            verdict_label = "Moderate non-constancy"
        else:
            verdict_str   = "WEAK"
            verdict_label = "Weak non-constancy"

    print(f"\n  Final verdict: >>> {verdict_str} <<<  ({verdict_label})")

    # ── 2. D_k STRUCTURE TABLE ────────────────────────────────────────────────
    print(f"\n{'2. D_k STRUCTURE  (D_k(t) = -s_k(t)/k²)':}")
    print(sep2)
    k_vals = SK_K_VALUES
    t_vals = SK_HORIZONS
    # Build matrix: rows=k, cols=t
    Dk_mat = np.full((len(k_vals), len(t_vals)), np.nan)
    for ki, k in enumerate(k_vals):
        for ti, t in enumerate(t_vals):
            entry = sk_raw_results.get(k, {}).get(t)
            if entry is not None:
                Dk_mat[ki, ti] = entry[1]   # index 1 = d_k

    # Header
    col_w = 9
    hdr = f"  {'k':>5}  " + "  ".join(f"{'t='+str(t):>{col_w}}" for t in t_vals)
    print(hdr)
    print("  " + "-" * (len(hdr) - 2))
    for ki, k in enumerate(k_vals):
        row_vals = Dk_mat[ki]
        row_str  = "  ".join(
            f"{v:>{col_w}.5f}" if np.isfinite(v) else f"{'nan':>{col_w}}"
            for v in row_vals
        )
        print(f"  {k:>5.1f}  {row_str}")

    # Mean across k for each t
    print()
    mean_dk = np.nanmean(Dk_mat, axis=0)
    min_dk  = np.nanmin(Dk_mat,  axis=0)
    max_dk  = np.nanmax(Dk_mat,  axis=0)
    delta_dk = max_dk - min_dk

    def _row(label, arr):
        s = "  ".join(
            f"{v:>{col_w}.5f}" if np.isfinite(v) else f"{'nan':>{col_w}}"
            for v in arr
        )
        print(f"  {label:>5}  {s}")

    _row("MEAN",  mean_dk)
    _row("MIN",   min_dk)
    _row("MAX",   max_dk)
    _row("ΔD_k",  delta_dk)

    print(f"\n  ΔD_k = max_k D_k(t) - min_k D_k(t)  per horizon:")
    for ti, t in enumerate(t_vals):
        flag_mark = " ← FLAGGED" if any(f["t"]==t for f in flagged) else ""
        dv = delta_dk[ti]
        print(f"    t={t:>4} :  ΔD_k = {dv:.6f}{flag_mark}")

    # ── 3. PRACTICAL SIGNIFICANCE CHECK ──────────────────────────────────────
    print(f"\n{'3. PRACTICAL SIGNIFICANCE CHECK':}")
    print(sep2)
    print(f"  Thresholds:")
    print(f"    SK_RELATIVE_CHANGE_THRESH = {SK_RELATIVE_CHANGE_THRESH}  (δ*=10%, Eq.21)")
    print(f"    SK_ABSOLUTE_CHANGE_THRESH = {SK_ABSOLUTE_CHANGE_THRESH}")
    print(f"    DK_PIVOT (sigmoid mid)    = {DK_PIVOT}")
    print(f"    DK_SCALE (sigmoid steep)  = {DK_SCALE}")
    print()
    max_delta = float(np.nanmax(delta_dk)) if np.any(np.isfinite(delta_dk)) else 0.0
    max_hdelta = float(np.nanmax([horizon_dk[t] for t in SK_HORIZONS]))
    print(f"  Max ΔD_k across all horizons:   {max_delta:.6f}")
    print(f"  Max horizon_dk (rel_change):     {max_hdelta:.4f}")
    print(f"  dk_estimated (raw signal):       {dk_estimated:.6f}")
    sig_weight = 1.0 / (1.0 + np.exp(-(dk_estimated - DK_PIVOT) / DK_SCALE))
    print(f"  sigmoid weight at dk_estimated:  {sig_weight:.4f}")
    print(f"  dk_applied = est × weight:       {dk_applied:.6f}")
    print()
    if max_hdelta < SK_RELATIVE_CHANGE_THRESH and max_delta < SK_ABSOLUTE_CHANGE_THRESH * 10:
        sig_level = "NEGLIGIBLE"
    elif max_hdelta < 0.30 and len(flagged) < 3:
        sig_level = "MODERATE"
    else:
        sig_level = "LARGE"
    print(f"  Variation in D_k: >>> {sig_level} <<<")
    print()
    print(f"  Continuous correction breakdown by alpha:")
    for alpha in ALPHAS:
        eta   = ETA_BY_ALPHA[alpha]
        corr  = 1.0 + eta * dk_applied
        print(f"    alpha={alpha}:  ETA={eta}  →  VaR multiplier = {corr:.5f}  "
              f"(+{(corr-1)*100:.2f}%)")

    # ── 4. VAR PERFORMANCE CONTEXT ────────────────────────────────────────────
    print(f"\n{'4. VaR PERFORMANCE CONTEXT':}")
    print(sep2)
    models = ["Historical", "Parametric", "GARCH", "MF-VaR", "MF-VaR-Dk"]
    for alpha in ALPHAS:
        evs = eval_by_alpha.get(alpha, {})
        print(f"\n  alpha={alpha}  (target breach rate = {alpha:.2f})")
        print(f"  {'Model':<14} {'breach_rate':>12} {'calib_err':>10} "
              f"{'kupiec_p':>10} {'kupiec_pass':>12} {'es_ratio':>10} {'breach_AC':>10}")
        print(f"  {'-'*14}  {'-'*12}  {'-'*10}  {'-'*10}  {'-'*12}  {'-'*10}  {'-'*10}")
        for m in models:
            ev = evs.get(m, {})
            br  = ev.get("breach_rate",     np.nan)
            ce  = ev.get("calibration_err", np.nan)
            kp  = ev.get("kupiec_p",        np.nan)
            kpass = "PASS" if ev.get("kupiec_pass") else "FAIL"
            esr = ev.get("es_ratio",        np.nan)
            bac = ev.get("breach_autocorr", np.nan)
            mf_well = (m == "MF-VaR" and ev.get("kupiec_pass"))
            tag = " ←" if mf_well else ""
            print(f"  {m:<14}  {br:>12.4f}  {ce:>10.4f}  {kp:>10.4f}  "
                  f"{kpass:>12}  {esr:>10.4f}  {bac:>10.4f}{tag}")

    mfvar_well_01 = eval_by_alpha.get(0.01, {}).get("MF-VaR", {}).get("kupiec_pass", False)
    mfvar_well_05 = eval_by_alpha.get(0.05, {}).get("MF-VaR", {}).get("kupiec_pass", False)
    print()
    print(f"  Is MF-VaR already well-calibrated?")
    print(f"    alpha=0.01: {'YES — Kupiec PASS' if mfvar_well_01 else 'NO  — Kupiec FAIL'}")
    print(f"    alpha=0.05: {'YES — Kupiec PASS' if mfvar_well_05 else 'NO  — Kupiec FAIL'}")

    # ── 5. dk ACTIVATION DECISION ────────────────────────────────────────────
    print(f"\n{'5. dk ACTIVATION DECISION':}")
    print(sep2)
    print(f"  ┌─ ALWAYS PRINTED — never suppressed ─────────────────────────┐")
    print(f"  │  dk_estimated  (raw, before weighting) = {dk_estimated:>10.6f}        │")
    print(f"  │  sigmoid_weight(dk_estimated)          = {sig_weight:>10.4f}        │")
    print(f"  │  dk_applied    (used in VaR formula)   = {dk_applied:>10.6f}        │")
    print(f"  └─────────────────────────────────────────────────────────────┘")
    print()

    nonconstancy_present = len(flagged) > 0
    material_effect      = sig_level in ("MODERATE","LARGE")
    dk_improves          = True  # continuous correction always non-negative

    print(f"  Decision logic:")
    print(f"    Non-constancy present?              {'YES' if nonconstancy_present else 'NO'}")
    print(f"    Materially affects risk estimates?  {'YES' if material_effect else 'NO'}")
    print(f"    Does dk improve calibration?        {'YES (continuous — always helps proportionally)' if dk_applied > 1e-6 else 'NEGLIGIBLE EFFECT'}")
    print()
    print(f"  OLD v5 binary logic would have:")
    old_applied = dk_estimated if (0.01 in ALPHAS and dk_estimated > 0.05) else 0.0
    print(f"    dk_applied (binary, alpha=0.01) = {old_applied:.6f}")
    print(f"    dk_applied (binary, alpha=0.05) = 0.000000  [hard-gated]")
    print()
    print(f"  NEW continuous logic:")
    for alpha in ALPHAS:
        eta  = ETA_BY_ALPHA[alpha]
        corr = 1.0 + eta * dk_applied
        print(f"    alpha={alpha}:  dk_applied={dk_applied:.6f}  "
              f"ETA={eta}  →  multiplier={corr:.5f}")
    print()
    # MANDATORY final line
    print(f"  → dk_estimated = {dk_estimated:.6f}   (NOT zero — raw computed candidate)")
    print(f"  → dk_applied   = {dk_applied:.6f}   (sigmoid-weighted, used in VaR)")

    # ── 6. COUNTERFACTUAL ────────────────────────────────────────────────────
    print(f"\n{'6. COUNTERFACTUAL':}")
    print(sep2)
    dk_forced = max(dk_estimated, 1.0)   # force full-scale dk
    weight_forced = 1.0 / (1.0 + np.exp(-(dk_forced - DK_PIVOT) / DK_SCALE))
    dk_applied_forced = dk_forced * weight_forced
    print(f"  Scenario: dk FORCED ON at full scale (dk=1.0)")
    print(f"    dk_forced                  = {dk_forced:.4f}")
    print(f"    sigmoid_weight(dk_forced)  = {weight_forced:.4f}")
    print(f"    dk_applied_forced          = {dk_applied_forced:.4f}")
    print()
    for alpha in ALPHAS:
        eta    = ETA_BY_ALPHA[alpha]
        corr_actual = 1.0 + eta * dk_applied
        corr_forced = 1.0 + eta * dk_applied_forced
        pct_delta   = (corr_forced - corr_actual) / corr_actual * 100
        ev = eval_by_alpha.get(alpha, {})
        mfvar_br  = ev.get("MF-VaR",    {}).get("breach_rate", np.nan)
        mfdk_br   = ev.get("MF-VaR-Dk", {}).get("breach_rate", np.nan)
        print(f"  alpha={alpha}:")
        print(f"    Current  multiplier = {corr_actual:.5f}")
        print(f"    Forced   multiplier = {corr_forced:.5f}  (Δ = +{pct_delta:.2f}%)")
        print(f"    MF-VaR   breach_rate (actual)      = {mfvar_br:.4f}")
        print(f"    MF-VaR-Dk breach_rate (actual dk)  = {mfdk_br:.4f}")
        forced_br_est = mfvar_br * (corr_actual / corr_forced) if np.isfinite(mfvar_br) else np.nan
        if np.isfinite(forced_br_est):
            delta_br = mfvar_br - forced_br_est
            direction = "IMPROVE (fewer breaches)" if delta_br > 0 else "WORSEN (more breaches)"
            print(f"    Estimated breach_rate if forced  = {forced_br_est:.4f}  "
                  f"→ violations would {direction}")

    # ── 7. INTERPRETATION ────────────────────────────────────────────────────
    print(f"\n{'7. INTERPRETATION':}")
    print(sep2)

    # Gather signals for interpretation
    mfvar_br_01   = eval_by_alpha.get(0.01,{}).get("MF-VaR",{}).get("breach_rate", np.nan)
    mfvar_br_05   = eval_by_alpha.get(0.05,{}).get("MF-VaR",{}).get("breach_rate", np.nan)
    mfdk_br_01    = eval_by_alpha.get(0.01,{}).get("MF-VaR-Dk",{}).get("breach_rate", np.nan)
    hist_br_01    = eval_by_alpha.get(0.01,{}).get("Historical",{}).get("breach_rate", np.nan)
    param_br_01   = eval_by_alpha.get(0.01,{}).get("Parametric",{}).get("breach_rate", np.nan)
    garch_br_01   = eval_by_alpha.get(0.01,{}).get("GARCH",{}).get("breach_rate", np.nan)
    mfvar_kp_01   = eval_by_alpha.get(0.01,{}).get("MF-VaR",{}).get("kupiec_pass", False)
    mfdk_kp_01    = eval_by_alpha.get(0.01,{}).get("MF-VaR-Dk",{}).get("kupiec_pass", False)
    mfvar_esr_01  = eval_by_alpha.get(0.01,{}).get("MF-VaR",{}).get("es_ratio", np.nan)
    mfvar_bac_01  = eval_by_alpha.get(0.01,{}).get("MF-VaR",{}).get("breach_autocorr", np.nan)
    param_kp_01   = eval_by_alpha.get(0.01,{}).get("Parametric",{}).get("kupiec_pass", False)
    garch_kp_01   = eval_by_alpha.get(0.01,{}).get("GARCH",{}).get("kupiec_pass", False)

    print(f"  Asset characteristics:")
    print(f"    H_mid_mean   = {H_mid_mean:.4f}  "
          f"({'persistent' if H_mid_mean>0.55 else 'antipersistent' if H_mid_mean<0.45 else 'near-random walk'})")
    print(f"    d_alpha_mean = {d_alpha_mean:.4f}  "
          f"({'rich multifractal structure' if d_alpha_mean>1.0 else 'moderate' if d_alpha_mean>0.5 else 'near-monofractal'})")
    print(f"    rho_min_mean = {rho_min_mean:.4f}  "
          f"({'strong loss/gain asymmetry' if rho_min_mean<0.5 else 'moderate' if rho_min_mean<0.8 else 'symmetric'})")
    print(f"    dk_signal    = {dk_estimated:.4f}  "
          f"({'strong cumulant structure' if dk_estimated>0.20 else 'moderate' if dk_estimated>0.05 else 'near-Gaussian cumulants'})")
    print()

    # Classify: structural non-Gaussianity vs model misspecification
    # "Structural non-Gaussianity without financial impact":
    #   - dk high BUT MF-VaR already passes Kupiec
    #   - Adding dk doesn't change breach rate materially
    #   - ES ratio stays near 1.0
    # "True model misspecification":
    #   - Parametric & GARCH both fail
    #   - MF-VaR fails or barely passes
    #   - dk correction improves Kupiec outcome
    #   - High breach AC (clustering)

    all_trad_fail   = (not param_kp_01) and (not garch_kp_01)
    mfvar_pass      = mfvar_kp_01
    mfdk_pass       = mfdk_kp_01
    dk_helps        = mfdk_pass and not mfvar_pass
    high_bac        = np.isfinite(mfvar_bac_01) and abs(mfvar_bac_01) > 0.10
    high_dk         = dk_estimated > 0.10
    esr_off         = np.isfinite(mfvar_esr_01) and abs(mfvar_esr_01 - 1.0) > 0.20

    if high_dk and mfvar_pass and not dk_helps:
        classification = "STRUCTURAL NON-GAUSSIANITY WITHOUT FINANCIAL IMPACT"
        explanation = (
            f"    The asset shows measurable higher-order cumulant structure\n"
            f"    (dk_estimated={dk_estimated:.4f}) but MF-VaR already captures\n"
            f"    the risk adequately — Kupiec PASSES without dk correction.\n"
            f"    The non-Gaussianity is real but already absorbed by H_mid /\n"
            f"    d_alpha / rho_min factors. The dk term adds precision without\n"
            f"    correcting a fundamental failure."
        )
    elif all_trad_fail and (not mfvar_pass or dk_helps):
        classification = "TRUE RISK MODEL MISSPECIFICATION"
        explanation = (
            f"    All traditional models fail Kupiec (param={not param_kp_01},\n"
            f"    garch={not garch_kp_01}). The multifractal correction is\n"
            f"    materially necessary. The dk signal (={dk_estimated:.4f}) captures\n"
            f"    a genuine deficiency in the volatility model — the cumulant\n"
            f"    structure of displacements is non-constant in a way that\n"
            f"    produces systematic breach clustering (AC={mfvar_bac_01:.3f}).\n"
            f"    Ignoring dk would leave VaR structurally under-estimated."
        )
    elif high_dk and esr_off:
        classification = "PARTIAL MISSPECIFICATION — TAIL UNDERESTIMATION"
        explanation = (
            f"    The model calibrates breach frequency (Kupiec) but the\n"
            f"    Expected Shortfall ratio ({mfvar_esr_01:.3f}) deviates from 1.0,\n"
            f"    indicating the tail shape is misspecified even when breach\n"
            f"    count is correct. The dk structure (={dk_estimated:.4f}) captures\n"
            f"    non-Gaussian cumulants that affect loss severity, not just\n"
            f"    breach count. This is a severity misspecification."
        )
    elif high_dk and high_bac:
        classification = "REGIME-DRIVEN MISSPECIFICATION — CLUSTERING PRESENT"
        explanation = (
            f"    Breach autocorrelation ({mfvar_bac_01:.3f}) indicates clustered\n"
            f"    losses. The dk signal (={dk_estimated:.4f}) reflects non-constant\n"
            f"    displacement cumulants that correspond to regime transitions.\n"
            f"    The correction is warranted to reduce clustering, even if\n"
            f"    total breach count is near-correct."
        )
    else:
        classification = "NEAR-GAUSSIAN / WELL-SPECIFIED"
        explanation = (
            f"    Low dk_estimated ({dk_estimated:.4f}), MF-VaR passes (or near-passes)\n"
            f"    Kupiec. The asset's return distribution is adequately captured\n"
            f"    by the three-factor MF model. No evidence of systematic\n"
            f"    cumulant misspecification requiring dk correction."
        )

    print(f"  Classification: >>> {classification} <<<")
    print()
    print(f"  Reasoning:")
    print(explanation)
    print()
    print(f"  Summary evidence:")
    print(f"    All traditional models fail?    {'YES' if all_trad_fail else 'NO'}")
    print(f"    MF-VaR Kupiec pass (0.01)?      {'YES' if mfvar_pass else 'NO'}")
    print(f"    dk correction helps (0.01)?     {'YES' if dk_helps else 'NO'}")
    print(f"    High breach clustering?         {'YES' if high_bac else 'NO'} "
          f"(AC={mfvar_bac_01:.3f})")
    print(f"    ES ratio significantly off?     {'YES' if esr_off else 'NO'} "
          f"(ESR={mfvar_esr_01:.3f})")
    print(f"    Strong dk signal?               {'YES' if high_dk else 'NO'} "
          f"(dk={dk_estimated:.4f})")
    print()
    print(sep)


# ═══════════════════════════════════════════════════════════════════════════════
#  DATA
# ═══════════════════════════════════════════════════════════════════════════════
def batch_download():
    print(f"Downloading {len(ALL_TICKERS)} tickers...")
    df = yf.download(ALL_TICKERS, start=START, end=END, progress=False,
                     auto_adjust=True, group_by="ticker")
    result = {}
    for ticker in ALL_TICKERS:
        try:
            close = (df["Close"].dropna().values.ravel() if len(ALL_TICKERS)==1
                     else df[ticker]["Close"].dropna().values.ravel()).astype(float)
            if len(close) < 500: continue
            result[ticker] = np.diff(np.log(close[close > 0]))
        except Exception:
            pass
    print(f"  Got {len(result)}/{len(ALL_TICKERS)} tickers")
    return result


# ═══════════════════════════════════════════════════════════════════════════════
#  MF-DFA ENGINE
# ═══════════════════════════════════════════════════════════════════════════════
def _integrate(x): return np.cumsum(x - x.mean())

def mfdfa_vectorized(x, q_vals, s_min=10, s_max=None, n_scales=10):
    y=_integrate(x); n=len(y)
    if s_max is None: s_max=n//4
    s_max=min(s_max,n//4); s_min=max(s_min,4)
    if s_max<=s_min: return np.full(len(q_vals),np.nan)
    scales=np.unique(np.logspace(np.log10(s_min),np.log10(s_max),n_scales).astype(int))
    fqs=np.full((len(q_vals),len(scales)),np.nan)
    for si,s in enumerate(scales):
        step=max(1,s//4); starts=np.arange(0,n-s+1,step)
        if len(starts)<3: continue
        idx=starts[:,None]+np.arange(s); segs=y[idx]
        t=np.arange(s,dtype=float); tn=t-t.mean()
        denom=(tn**2).sum()
        if denom==0: continue
        slopes=(segs*tn).sum(1)/denom
        fitted=slopes[:,None]*t+(segs.mean(1)-slopes*t.mean())[:,None]
        rms=np.sqrt(((segs-fitted)**2).mean(1)); rms=rms[rms>0]
        if len(rms)<2: continue
        for j,q in enumerate(q_vals):
            fqs[j,si]=(np.exp(0.5*np.mean(np.log(rms**2))) if q==0
                       else (np.mean(rms**q))**(1./q))
    hq=np.full(len(q_vals),np.nan); log_s=np.log(scales.astype(float))
    for j in range(len(q_vals)):
        row=fqs[j]; mask=np.isfinite(row)&(row>0)
        if mask.sum()>=4: hq[j]=np.polyfit(log_s[mask],np.log(row[mask]),1)[0]
    return hq

def compute_rho_curve(x, q_pos, s_min=10, s_max=None, n_scales=8):
    y=_integrate(x); n=len(y)
    if s_max is None: s_max=n//4
    s_max=min(s_max,n//4); s_min=max(s_min,3)
    if s_max<=s_min: return np.full(len(q_pos),np.nan)
    scales=np.unique(np.logspace(np.log10(s_min),np.log10(s_max),n_scales).astype(int))
    rho_curve=np.full(len(q_pos),np.nan)
    for j,q in enumerate(q_pos):
        fp,fn=[],[]
        for s in scales:
            step=max(1,s//4); starts=np.arange(0,n-s+1,step)
            if len(starts)<3: continue
            idx=starts[:,None]+np.arange(s); segs=y[idx]
            t=np.arange(s,dtype=float); tn=t-t.mean()
            denom=(tn**2).sum()
            if denom==0: continue
            slopes=(segs*tn).sum(1)/denom
            fitted=slopes[:,None]*t+(segs.mean(1)-slopes*t.mean())[:,None]
            rms=np.sqrt(((segs-fitted)**2).mean(1)); rms=rms[rms>0]
            if len(rms)<2: continue
            fp.append((np.mean(rms** q))**(1.0/ q))
            fn.append((np.mean(rms**-q))**(1.0/-q))
        if len(fp)<3: continue
        a=np.log(np.array(fp)); b=np.log(np.array(fn))
        mask=np.isfinite(a)&np.isfinite(b)
        if mask.sum()<3: continue
        a,b=a[mask],b[mask]
        cov=np.mean((a-a.mean())*(b-b.mean()))
        den=a.std()*b.std()
        if den>0: rho_curve[j]=cov/den
    return rho_curve

def extract_mf_risk_features(returns):
    n=len(returns); q_pos=np.arange(1,31,dtype=float)
    H_mid=np.full(n,np.nan); d_alpha=np.full(n,np.nan)
    rho_min=np.full(n,np.nan); rho_slope=np.full(n,np.nan)
    for t in range(MF_WINDOW,n,MF_STEP):
        w=returns[t-MF_WINDOW:t]
        hq=mfdfa_vectorized(w,Q_VALS,s_min=10,s_max=MF_WINDOW//4,n_scales=10)
        valid=np.isfinite(hq)
        if valid.sum()>=4:
            mid=len(Q_VALS)//2
            H_mid[t]=hq[mid] if np.isfinite(hq[mid]) else np.nanmean(hq[valid])
            tau=Q_VALS*hq-1.
            wlen=min(5,len(tau)-(1 if len(tau)%2==0 else 0))
            tau_s=savgol_filter(tau,wlen,2) if wlen>=3 else tau
            af=np.gradient(tau_s,Q_VALS); af=af[np.isfinite(af)]
            if len(af)>=2: d_alpha[t]=af.max()-af.min()
        rc=compute_rho_curve(w,q_pos,s_min=10,s_max=MF_WINDOW//4,n_scales=8)
        valid_rc=np.isfinite(rc)
        if valid_rc.sum()>=3:
            rho_min[t]=float(np.nanmin(rc))
            q_v=q_pos[valid_rc]; r_v=rc[valid_rc]
            if len(q_v)>=2: rho_slope[t]=float(np.polyfit(q_v,r_v,1)[0])
    for arr in [H_mid,d_alpha,rho_min,rho_slope]:
        last=np.nan
        for i in range(len(arr)):
            if np.isfinite(arr[i]): last=arr[i]
            elif np.isfinite(last): arr[i]=last
        fv=arr[np.isfinite(arr)]
        if len(fv): arr[~np.isfinite(arr)]=fv[0]
        else:       arr[:]=0.
    return H_mid,d_alpha,rho_min,rho_slope


# ═══════════════════════════════════════════════════════════════════════════════
#  s_k / D_k ENGINE   — paper Eq. 19: Δr(τ,t) = r(τ+t) − r(τ)
# ═══════════════════════════════════════════════════════════════════════════════
def _sk_single(disps, k, t):
    if len(disps) < 5: return np.nan, np.nan
    cf  = np.mean(np.exp(1j * k * disps))
    mag = np.abs(cf)
    if mag < 1e-15: return np.nan, mag
    return (1.0 / t) * np.log(mag), mag

def _run_sk_analysis(disps_by_t):
    results = {}
    for k in SK_K_VALUES:
        results[k] = {}
        for t, disps in disps_by_t.items():
            sk, mag    = _sk_single(disps, k, t)
            d_k        = (-sk / k**2) if (sk is not None and not np.isnan(sk)) else np.nan
            sk_over_k2 = (sk / k**2)  if (sk is not None and not np.isnan(sk)) else np.nan
            results[k][t] = (sk, d_k, sk_over_k2, len(disps),
                             mag if mag is not None else np.nan)
    return results

def _detect_nonconstancy(results, horizons):
    flagged   = []
    n_k_total = len(SK_K_VALUES)
    k_arr     = np.array(SK_K_VALUES)
    for t in horizons:
        sk2_arr = np.array([results[k][t][2] for k in SK_K_VALUES])
        mag_arr = np.array([results[k][t][4] for k in SK_K_VALUES])
        reliable = (~np.isnan(sk2_arr)) & (~np.isnan(mag_arr)) & (mag_arr >= SK_CF_MIN_MAG)
        vals     = sk2_arr[reliable]
        k_rel    = k_arr[reliable]
        n_rel    = int(reliable.sum())
        if n_rel < SK_MIN_VALID_K: continue
        if n_rel / n_k_total < SK_VALIDITY_RATIO: continue
        min_val    = float(np.min(vals))
        max_val    = float(np.max(vals))
        abs_change = max_val - min_val
        mean_abs   = float(np.mean(np.abs(vals)))
        if mean_abs > SK_MIN_ABS_MEAN:
            rel_change  = abs_change / mean_abs
            change_type = "relative"
        else:
            rel_change  = abs_change / SK_ABSOLUTE_CHANGE_THRESH
            change_type = "absolute"
        spearman_r, spearman_p = np.nan, np.nan
        if n_rel >= 4:
            spearman_r, spearman_p = spearmanr(k_rel, vals)
        should_flag = ((change_type == "relative" and rel_change > SK_RELATIVE_CHANGE_THRESH) or
                       (change_type == "absolute" and abs_change > SK_ABSOLUTE_CHANGE_THRESH))
        if should_flag:
            flagged.append({
                "t":          t,
                "n_reliable": n_rel,
                "coverage":   n_rel / n_k_total,
                "mean_sk_k2": float(np.mean(vals)),
                "abs_change": abs_change,
                "rel_change": rel_change if change_type == "relative" else np.nan,
                "spearman_r": spearman_r,
                "spearman_p": spearman_p,
                "direction":  "increasing" if max_val > min_val else "decreasing",
            })
    return flagged

def _sigmoid(x, pivot=DK_PIVOT, scale=DK_SCALE):
    """Sigmoid weighting function for continuous dk correction."""
    return 1.0 / (1.0 + np.exp(-(x - pivot) / scale))

def compute_sk_nonconstancy(returns):
    """
    Full s_k / D_k pipeline.

    Returns
    -------
    dk_estimated   : float — raw aggregate non-constancy (0 = constant D_k)
    dk_applied     : float — sigmoid-weighted value used in VaR correction
    flagged        : list  — flag dicts per horizon
    horizon_dk     : dict  — {t: rel_change} for all SK_HORIZONS
    sk_raw_results : dict  — {k: {t: (sk, d_k, sk_over_k2, n, mag)}}
    """
    n = len(returns)
    disps_by_t = {}
    for t in SK_HORIZONS:
        disps = np.array([
            returns[i + t] - returns[i]       # paper Eq. 19
            for i in range(SK_N_ORIGINS)
            if i + t < n
        ])
        disps_by_t[t] = disps if len(disps) >= 5 else np.array([])

    sk_raw_results = _run_sk_analysis(disps_by_t)
    flagged        = _detect_nonconstancy(sk_raw_results, SK_HORIZONS)

    flagged_set = {f["t"]: f for f in flagged}
    horizon_dk  = {}
    for t in SK_HORIZONS:
        if t in flagged_set:
            fd  = flagged_set[t]
            val = fd["rel_change"] if not np.isnan(fd["rel_change"]) else (
                fd["abs_change"] / SK_ABSOLUTE_CHANGE_THRESH)
            horizon_dk[t] = float(np.clip(val, 0., 5.))
        else:
            horizon_dk[t] = 0.

    # Aggregate over short-term flagged horizons (t≤20, per paper)
    # FIXED: Require MIN_FLAGGED_HORIZONS flags to activate
    MIN_FLAGGED_HORIZONS = 2  # At least 2 horizons must show non-constancy

    short_flagged = [horizon_dk[t] for t in SK_HORIZONS
                    if t <= 20 and horizon_dk[t] > 0]

    if len(short_flagged) >= MIN_FLAGGED_HORIZONS:
        dk_estimated = float(np.max(short_flagged))  # or use np.mean
    else:
        dk_estimated = 0.0

    # Continuous sigmoid weighting (NEW — replaces binary gate)
    sig_weight = _sigmoid(dk_estimated)
    dk_applied = dk_estimated * sig_weight

    return dk_estimated, dk_applied, flagged, horizon_dk, sk_raw_results


# ═══════════════════════════════════════════════════════════════════════════════
#  TRADITIONAL VAR MODELS
# ═══════════════════════════════════════════════════════════════════════════════
def historical_var(returns, alpha, window=HIST_VAR_WINDOW):
    n=len(returns); var=np.full(n,np.nan)
    for t in range(window,n):
        var[t]=-np.percentile(returns[t-window:t],alpha*100)
    return var

def parametric_var(returns, alpha, window=PARAM_VAR_WINDOW):
    n=len(returns); var=np.full(n,np.nan); z=stats.norm.ppf(alpha)
    for t in range(window,n):
        w=returns[t-window:t]; var[t]=-(w.mean()+z*(w.std()+1e-10))
    return var

def _ewma_var(returns, alpha, lam=EWMA_LAMBDA):
    n=len(returns); var=np.full(n,np.nan); z=stats.norm.ppf(alpha)
    sig2=returns[:20].var()+1e-10
    for t in range(1,n):
        sig2=lam*sig2+(1-lam)*returns[t-1]**2
        mu=returns[max(0,t-20):t].mean()
        var[t]=-(mu+z*np.sqrt(sig2))
    return var

def garch_var(returns, alpha, train_n):
    n=len(returns); var=np.full(n,np.nan); z=stats.norm.ppf(alpha)
    if HAS_ARCH:
        r_train=returns[:train_n]*100
        try:
            am=arch_model(r_train,vol='Garch',p=1,q=1,dist='Normal',rescale=False)
            res=am.fit(disp='off',show_warning=False)
            omega=res.params['omega']
            alpha_g=res.params['alpha[1]']
            beta_g=res.params['beta[1]']
            sig2=float(res.conditional_volatility[-1]**2)
            for t in range(train_n,n):
                r_prev=(returns[t-1]*100)**2
                sig2=omega+alpha_g*r_prev+beta_g*sig2
                sig=np.sqrt(sig2)/100
                mu=returns[max(0,t-20):t].mean()
                var[t]=-(mu+z*sig)
            return var
        except Exception:
            pass
    return _ewma_var(returns,alpha)

def _compute_ewma_vol(returns, lam=EWMA_LAMBDA):
    n=len(returns); sig2=returns[:20].var()+1e-10
    ewma_vol=np.full(n,np.nan)
    for t in range(1,n):
        sig2=lam*sig2+(1-lam)*returns[t-1]**2
        ewma_vol[t]=np.sqrt(sig2)
    return ewma_vol


# ═══════════════════════════════════════════════════════════════════════════════
#  MF-VAR  (unchanged three-factor core)
# ═══════════════════════════════════════════════════════════════════════════════
def mf_var_core(returns, alpha, H_mid, d_alpha, rho_min, window=PARAM_VAR_WINDOW):
    p=MF_PARAMS[alpha]
    n=len(returns); z=stats.norm.ppf(alpha); var=np.full(n,np.nan)
    for t in range(window,n):
        w=returns[t-window:t]; mu=w.mean(); sig=w.std()+1e-10
        base=-(mu+z*sig)
        h  =float(np.clip(H_mid[t],0.1,0.9))
        da =float(np.clip(d_alpha[t],0.,2.))
        rho=float(np.clip(rho_min[t],-1.,1.))
        var[t]=(base
                *(h/0.5)**p["gamma"]
                *(1.+p["beta"]*da)
                *(1.+p["delta"]*(1.-rho)))
    return var

def mf_var(returns, alpha, H_mid, d_alpha, rho_min, window=PARAM_VAR_WINDOW):
    mf_raw   = mf_var_core(returns,alpha,H_mid,d_alpha,rho_min,window)
    param_raw= parametric_var(returns,alpha,window)
    ewma_vol = _compute_ewma_vol(returns)
    lo,hi    = MF_BLEND_LOW_VOL, MF_BLEND_HIGH_VOL
    w_mf     = np.clip((ewma_vol-lo)/(hi-lo), 0., 1.)
    blended  = np.where(
        np.isfinite(mf_raw) & np.isfinite(param_raw),
        w_mf*mf_raw + (1.-w_mf)*param_raw,
        mf_raw
    )
    return blended

def mf_var_dk(returns, alpha, H_mid, d_alpha, rho_min, dk_applied,
              window=PARAM_VAR_WINDOW):
    """
    MF-VaR with CONTINUOUS s_k non-constancy correction.

    CHANGED FROM v5 BINARY GATE:
      Old: if alpha != 0.01 or dk < threshold: return base (zero correction)
      New: correction = 1 + ETA_BY_ALPHA[alpha] * dk_applied
           where dk_applied = dk_estimated * sigmoid(dk_estimated)
           Applied at ALL alpha levels, scaled proportionally.

    This ensures:
      - No discontinuity at the threshold boundary
      - Small signals produce small corrections (not zero)
      - alpha=0.05 gets a correction too, just with smaller ETA
      - dk_applied is always logged (never silently zeroed)
    """
    base = mf_var(returns, alpha, H_mid, d_alpha, rho_min, window)
    eta  = ETA_BY_ALPHA[alpha]
    dk_correction = 1.0 + eta * float(np.clip(dk_applied, 0., 5.))
    return np.where(np.isfinite(base), base * dk_correction, np.nan)

def apply_breach_feedback(var_series, returns_test):
    out = var_series.copy(); n = len(out)
    for t in range(1, n):
        if (np.isfinite(out[t-1]) and np.isfinite(returns_test[t-1])
                and returns_test[t-1] < -out[t-1]):
            if np.isfinite(out[t]):
                out[t] = out[t] * (1. + MF_SHOCK_SCALE)
    return out


# ═══════════════════════════════════════════════════════════════════════════════
#  EVALUATION METRICS
# ═══════════════════════════════════════════════════════════════════════════════
def kupiec_pof(breaches, n, alpha):
    x=int(breaches.sum())
    if x==0 or x==n: return np.nan,np.nan,x/n
    p_hat=x/n
    lr=-2*(x*np.log(alpha/p_hat)+(n-x)*np.log((1-alpha)/(1-p_hat)))
    p_val = float(1-stats.chi2.cdf(lr,df=1))
    return float(lr), p_val, float(p_hat)

def christoffersen_test(breaches):
    b=breaches.astype(int)
    n00=int(((b[:-1]==0)&(b[1:]==0)).sum())
    n01=int(((b[:-1]==0)&(b[1:]==1)).sum())
    n10=int(((b[:-1]==1)&(b[1:]==0)).sum())
    n11=int(((b[:-1]==1)&(b[1:]==1)).sum())
    if n01+n11==0 or n00+n10==0: return np.nan,np.nan
    pi_01=n01/(n00+n01+1e-10); pi_11=n11/(n10+n11+1e-10)
    pi=(n01+n11)/(n00+n01+n10+n11+1e-10)
    if pi<=0 or pi>=1 or pi_01<=0 or pi_11<=0: return np.nan,np.nan
    ll_r=(n00+n10)*np.log(1-pi)+(n01+n11)*np.log(pi)
    ll_u=(n00*np.log(max(1-pi_01,1e-10))+n01*np.log(max(pi_01,1e-10))+
          n10*np.log(max(1-pi_11,1e-10))+n11*np.log(max(pi_11,1e-10)))
    lr=-2*(ll_r-ll_u)
    return float(lr),float(1-stats.chi2.cdf(lr,df=1))

def evaluate_model(returns_test, var_series, alpha):
    mask=np.isfinite(var_series)
    if mask.sum()<100: return None
    r=returns_test[mask]; v=var_series[mask]; n=len(r)
    breaches=(r<-v)
    lr_pof,p_pof,breach_rate=kupiec_pof(breaches,n,alpha)
    lr_cc,p_cc=christoffersen_test(breaches)
    if breaches.sum()>0:
        es_real=float(-r[breaches].mean())
        es_ratio=es_real/(float(v[breaches].mean())+1e-10)
    else:
        es_real=es_ratio=np.nan
    non_breach=r>=-v
    cap_eff=float(v[non_breach].mean()) if non_breach.sum()>0 else np.nan
    b=breaches.astype(float)
    b_ac=(float(np.corrcoef(b[:-1],b[1:])[0,1])
          if (len(b)>10 and b.std()>1e-10) else np.nan)
    return {
        "n":               n,
        "breach_rate":     breach_rate,
        "target_alpha":    alpha,
        "calibration_err": abs(breach_rate-alpha),
        "kupiec_lr":       lr_pof,
        "kupiec_p":        p_pof,
        "kupiec_pass":     bool(p_pof>0.05) if np.isfinite(p_pof) else False,
        "cc_lr":           lr_cc,
        "cc_p":            p_cc,
        "cc_pass":         bool(p_cc>0.05) if np.isfinite(p_cc) else False,
        "es_ratio":        es_ratio,
        "capital_eff":     cap_eff,
        "breach_autocorr": b_ac,
    }


# ═══════════════════════════════════════════════════════════════════════════════
#  GPU-ACCELERATED OPTUNA CALIBRATION  (unchanged from v5)
# ═══════════════════════════════════════════════════════════════════════════════
def _precompute_for_calibration(all_data, tickers):
    print(f"[Precompute] Extracting features for {len(tickers)} tickers...")
    def _one(ticker):
        r=all_data[ticker].astype(np.float64)
        n=len(r); train_n=int(n*TRAIN_FRAC)
        H_mid,d_alpha,rho_min,_=extract_mf_risk_features(r)
        ewma_vol=_compute_ewma_vol(r)
        roll_mu =np.array([r[t-PARAM_VAR_WINDOW:t].mean()
                           for t in range(PARAM_VAR_WINDOW,n)],dtype=np.float32)
        roll_sig=np.array([r[t-PARAM_VAR_WINDOW:t].std()+1e-10
                           for t in range(PARAM_VAR_WINDOW,n)],dtype=np.float32)
        ts=train_n-PARAM_VAR_WINDOW
        return ticker,{
            "H_mid":    H_mid[PARAM_VAR_WINDOW:].astype(np.float32),
            "d_alpha":  d_alpha[PARAM_VAR_WINDOW:].astype(np.float32),
            "rho_min":  rho_min[PARAM_VAR_WINDOW:].astype(np.float32),
            "ewma_vol": ewma_vol[PARAM_VAR_WINDOW:].astype(np.float32),
            "roll_mu":  roll_mu,
            "roll_sig": roll_sig,
            "r_test":   r[train_n:].astype(np.float32),
            "ts":       ts,
        }
    results=Parallel(n_jobs=-1,prefer="threads")(
        delayed(_one)(t) for t in tickers)
    precomp=dict(results)
    print(f"[Precompute] Done — {len(precomp)} tickers ready.")
    return precomp

def _score_batch_vectorized(pre_list, alpha, gamma, beta_, delta):
    xp=cp if GPU_AVAILABLE else np
    z=float(stats.norm.ppf(alpha))
    lo,hi=MF_BLEND_LOW_VOL,MF_BLEND_HIGH_VOL
    total_loss=0.; valid_count=0
    for pre in pre_list:
        ts=pre["ts"]
        if GPU_AVAILABLE:
            roll_mu =cp.asarray(pre["roll_mu"][ts:])
            roll_sig=cp.asarray(pre["roll_sig"][ts:])
            H       =cp.asarray(pre["H_mid"][ts:])
            da      =cp.asarray(pre["d_alpha"][ts:])
            rho     =cp.asarray(pre["rho_min"][ts:])
            ewma    =cp.asarray(pre["ewma_vol"][ts:])
            r_test  =cp.asarray(pre["r_test"])
        else:
            roll_mu =pre["roll_mu"][ts:]
            roll_sig=pre["roll_sig"][ts:]
            H       =pre["H_mid"][ts:]
            da      =pre["d_alpha"][ts:]
            rho     =pre["rho_min"][ts:]
            ewma    =pre["ewma_vol"][ts:]
            r_test  =pre["r_test"]
        H  =xp.clip(H,  0.1, 0.9)
        da =xp.clip(da, 0.0, 2.0)
        rho=xp.clip(rho,-1.0,1.0)
        base    = -(roll_mu + z*roll_sig)
        mf_raw  = base*(H/0.5)**gamma*(1.+beta_*da)*(1.+delta*(1.-rho))
        w       = xp.clip((ewma-lo)/(hi-lo), 0., 1.)
        blended = w*mf_raw + (1.-w)*base
        T=min(len(blended),len(r_test))
        blended=blended[:T]; r_t=r_test[:T]
        fin=xp.isfinite(blended)
        n_fin=int(fin.sum().get()) if GPU_AVAILABLE else int(fin.sum())
        if n_fin<100: continue
        v=blended[fin]; r_v=r_t[fin]
        bvec=(r_v<-v).astype(xp.float32)
        breach_rate=float(bvec.mean().get()) if GPU_AVAILABLE else float(bvec.mean())
        cal_err=abs(breach_rate-alpha)
        x=int(bvec.sum().get()) if GPU_AVAILABLE else int(bvec.sum())
        if 0<x<n_fin:
            p_hat=x/n_fin
            lr_kup=-2.*(x*np.log(alpha/p_hat)+(n_fin-x)*np.log((1-alpha)/(1-p_hat)))
            p_kup=float(1-stats.chi2.cdf(lr_kup,df=1))
        else:
            p_kup=0.
        b_cpu=bvec.get() if GPU_AVAILABLE else np.asarray(bvec)
        ac=(float(np.corrcoef(b_cpu[:-1],b_cpu[1:])[0,1])
            if (len(b_cpu)>10 and b_cpu.std()>1e-10) else 0.)
        total_loss +=W_CAL*cal_err+W_KUP*max(0.,0.05-p_kup)+W_AC*abs(ac)
        valid_count+=1
    return total_loss/valid_count if valid_count>0 else 1.

def _build_study(name, n_startup=10):
    return optuna.create_study(
        study_name=name, direction="minimize",
        sampler=TPESampler(n_startup_trials=n_startup,multivariate=True,seed=42),
        pruner=MedianPruner(n_startup_trials=5,n_warmup_steps=0,interval_steps=1),
    )

def _optimise_single_alpha(precomp, tickers, alpha, gk, bk, dk_key,
                           n_trials, prior, batch_size):
    pre_all=[precomp[t] for t in tickers if t in precomp]
    batches=[pre_all[i:i+batch_size] for i in range(0,len(pre_all),batch_size)]
    def objective(trial):
        g=trial.suggest_float(gk,    *_BOUNDS[gk])
        b=trial.suggest_float(bk,    *_BOUNDS[bk])
        d=trial.suggest_float(dk_key,*_BOUNDS[dk_key])
        losses=[_score_batch_vectorized(bt,alpha,g,b,d) for bt in batches]
        mean_loss=float(np.mean([l for l in losses if np.isfinite(l)]))
        trial.report(mean_loss,step=0)
        if trial.should_prune(): raise optuna.TrialPruned()
        return mean_loss
    study=_build_study(f"mfv5final_alpha_{int(alpha*100)}")
    study.enqueue_trial({gk:prior[gk],bk:prior[bk],dk_key:prior[dk_key]})
    study.optimize(objective,n_trials=n_trials,show_progress_bar=True,n_jobs=1,
                   timeout=600)
    best=study.best_params
    n_pruned=sum(t.state==optuna.trial.TrialState.PRUNED for t in study.trials)
    print(f"  [alpha={alpha}] best={study.best_value:.5f}  "
          f"gamma={best[gk]:.4f}  beta={best[bk]:.4f}  delta={best[dk_key]:.4f}  "
          f"({len(study.trials)} trials, {n_pruned} pruned)")
    return best[gk],best[bk],best[dk_key]

def calibrate_mf_gpu(all_data, n_tickers=CALIB_N_TICKERS, n_trials=CALIB_N_TRIALS,
                     batch_size=CALIB_BATCH_SIZE):
    tickers=[t for t in ALL_TICKERS if t in all_data][:n_tickers]
    print(f"\n{'='*70}")
    print(f"  v5-FINAL GPU Calibration  (GPU={'YES' if GPU_AVAILABLE else 'NO'})")
    print(f"  Tickers: {len(tickers)}  Trials/stage: {n_trials}  Batch: {batch_size}")
    print(f"{'='*70}\n")
    t0=time.time()
    precomp=_precompute_for_calibration(all_data,tickers)
    print(f"\n[Optuna] Stage A — alpha=0.01, {n_trials} trials...")
    g99,b99,d99=_optimise_single_alpha(
        precomp,tickers,0.01,"g99","b99","d99",n_trials,_PRIOR_99,batch_size)
    print(f"\n[Optuna] Stage B — alpha=0.05, {n_trials} trials...")
    g95,b95,d95=_optimise_single_alpha(
        precomp,tickers,0.05,"g95","b95","d95",n_trials,_PRIOR_95,batch_size)
    best_params={
        0.01:{"gamma":g99,"beta":b99,"delta":d99},
        0.05:{"gamma":g95,"beta":b95,"delta":d95},
    }
    print(f"\n[Calibration] Done in {time.time()-t0:.1f}s")
    for a,p in best_params.items():
        print(f"  alpha={a}: gamma={p['gamma']:.4f}  beta={p['beta']:.4f}  "
              f"delta={p['delta']:.4f}")
    return best_params


# ═══════════════════════════════════════════════════════════════════════════════
#  PER-TICKER BENCHMARK
# ═══════════════════════════════════════════════════════════════════════════════
def benchmark_ticker(ticker, returns, verbose_diag=True):
    if returns is None or len(returns)<600: return None
    n=len(returns); train_n=int(n*TRAIN_FRAC); r_test=returns[train_n:]

    print(f"    MF...", end=" ", flush=True)
    H_mid,d_alpha,rho_min,_=extract_mf_risk_features(returns)

    print(f"sk...", end=" ", flush=True)
    (dk_estimated, dk_applied,
     flagged, horizon_dk, sk_raw_results) = compute_sk_nonconstancy(returns)

    # Build all VaR series
    var_hist  = historical_var(returns,0.01)[train_n:]
    var_param = parametric_var(returns,0.01)[train_n:]
    var_garch = garch_var(returns,0.01,train_n)[train_n:]

    eval_by_alpha = {}
    series_by_alpha = {}
    for alpha in ALPHAS:
        a_key = f"a{int(alpha*100):02d}"

        var_hist_a  = historical_var(returns,alpha)[train_n:]
        var_param_a = parametric_var(returns,alpha)[train_n:]
        var_garch_a = garch_var(returns,alpha,train_n)[train_n:]

        var_mf_full   = mf_var(returns,alpha,H_mid,d_alpha,rho_min)
        # NEW: dk_applied is continuous — pass it directly
        var_mfdk_full = mf_var_dk(returns,alpha,H_mid,d_alpha,rho_min,dk_applied)

        var_mf_t   = apply_breach_feedback(var_mf_full[train_n:],   r_test)
        var_mfdk_t = apply_breach_feedback(var_mfdk_full[train_n:], r_test)

        models = {
            "Historical": var_hist_a,
            "Parametric":  var_param_a,
            "GARCH":       var_garch_a,
            "MF-VaR":      var_mf_t,
            "MF-VaR-Dk":   var_mfdk_t,
        }
        eval_by_alpha[alpha] = {}
        for name, var_series in models.items():
            ev = evaluate_model(r_test, var_series, alpha)
            if ev:
                eval_by_alpha[alpha][name] = ev
        series_by_alpha[alpha] = {k: v[-500:] for k,v in models.items()}

    # Full 7-section diagnostic printout
    if verbose_diag:
        print_diagnostic(
            ticker         = ticker,
            returns        = returns,
            dk_estimated   = dk_estimated,
            dk_applied     = dk_applied,
            flagged        = flagged,
            horizon_dk     = horizon_dk,
            sk_raw_results = sk_raw_results,
            eval_by_alpha  = eval_by_alpha,
            H_mid_mean     = float(np.nanmean(H_mid[train_n:])),
            d_alpha_mean   = float(np.nanmean(d_alpha[train_n:])),
            rho_min_mean   = float(np.nanmean(rho_min[train_n:])),
        )
    else:
        print("done.")

    result = {
        "ticker":        ticker,
        "dk_estimated":  dk_estimated,    # raw signal — always stored
        "dk_applied":    dk_applied,      # sigmoid-weighted — used in VaR
        "dk_n_flagged":  len(flagged),
        "H_mid_mean":    float(np.nanmean(H_mid[train_n:])),
        "d_alpha_mean":  float(np.nanmean(d_alpha[train_n:])),
        "rho_min_mean":  float(np.nanmean(rho_min[train_n:])),
        "horizon_dk":    horizon_dk,
        "eval_by_alpha": eval_by_alpha,
    }
    for alpha in ALPHAS:
        a_key = f"a{int(alpha*100):02d}"
        result[a_key] = eval_by_alpha.get(alpha, {})
        result[a_key]["_series"] = {
            "returns": r_test[-500:],
            **series_by_alpha[alpha],
        }
    return result


# ═══════════════════════════════════════════════════════════════════════════════
#  AGGREGATE
# ═══════════════════════════════════════════════════════════════════════════════
def aggregate_results(all_results, alpha):
    a_key=f"a{int(alpha*100):02d}"
    models=["Historical","Parametric","GARCH","MF-VaR","MF-VaR-Dk"]
    metrics=["calibration_err","kupiec_pass","cc_pass",
             "es_ratio","capital_eff","breach_autocorr"]
    agg={m:{k:[] for k in metrics} for m in models}
    for r in all_results:
        if r is None or a_key not in r: continue
        for m in models:
            if m not in r[a_key]: continue
            ev=r[a_key][m]
            for k in metrics:
                v=ev.get(k)
                if v is not None and np.isfinite(float(v)):
                    agg[m][k].append(float(v))
    rows=[]
    for m in models:
        row={"Model":m}
        for k in metrics:
            vals=agg[m][k]
            row[k+"_mean"]=float(np.mean(vals)) if vals else np.nan
            if k in ["kupiec_pass","cc_pass"]:
                row[k+"_rate"]=float(np.mean(vals)) if vals else np.nan
        rows.append(row)
    return pd.DataFrame(rows)


# ═══════════════════════════════════════════════════════════════════════════════
#  PLOTTING
# ═══════════════════════════════════════════════════════════════════════════════
def apply_dark(fig):
    fig.patch.set_facecolor(DARK["fig"])
    for ax in fig.get_axes():
        ax.set_facecolor(DARK["ax"])
        for sp in ax.spines.values(): sp.set_edgecolor(DARK["spine"])
        ax.tick_params(colors=DARK["text"])
        ax.xaxis.label.set_color(DARK["muted"])
        ax.yaxis.label.set_color(DARK["muted"])
        ax.title.set_color(DARK["text"])
        for lb in ax.get_xticklabels()+ax.get_yticklabels():
            lb.set_color(DARK["text"])
        leg=ax.get_legend()
        if leg:
            leg.get_frame().set_facecolor(DARK["ax"])
            leg.get_frame().set_edgecolor(DARK["spine"])
            for t in leg.get_texts(): t.set_color(DARK["text"])
    if fig._suptitle: fig._suptitle.set_color(DARK["text"])
    return fig

def save_dark(fig, path, dpi=150):
    apply_dark(fig)
    fig.savefig(path,dpi=dpi,bbox_inches="tight",facecolor=fig.get_facecolor())
    plt.close(fig)
    print(f"  Saved: {path}")


def plot_continuous_dk_curve(out_dir):
    """Visualise the sigmoid weighting function for dk correction."""
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    x = np.linspace(0, 1.0, 400)
    w = 1.0 / (1.0 + np.exp(-(x - DK_PIVOT) / DK_SCALE))
    dk_app = x * w

    ax = axes[0]
    ax.plot(x, w, color=DARK["blue"], lw=2, label="sigmoid weight")
    ax.axvline(DK_PIVOT, color=DARK["amber"], lw=1, ls="--",
               label=f"DK_PIVOT={DK_PIVOT}")
    ax.axhline(0.5, color=DARK["muted"], lw=0.8, ls=":")
    ax.set_xlabel("dk_estimated (raw signal)")
    ax.set_ylabel("sigmoid weight")
    ax.set_title("Sigmoid weighting function\n"
                 "dk_weight = σ((dk_est − pivot) / scale)",
                 fontsize=10, fontweight="bold")
    ax.legend(fontsize=9); ax.grid(True, alpha=0.3)
    ax.set_xlim(0, 1)

    ax = axes[1]
    ax.plot(x, dk_app, color=DARK["green"], lw=2, label="dk_applied = est × weight")
    ax.plot(x, x, color=DARK["muted"], lw=1, ls="--", label="dk_applied = est (no weighting)")
    for alpha in ALPHAS:
        eta = ETA_BY_ALPHA[alpha]
        corr = 1.0 + eta * dk_app
        ax.plot(x, (corr - 1)*100, lw=1.5,
                label=f"VaR bump % (alpha={alpha}, ETA={eta})",
                color=DARK["purple"] if alpha==0.01 else DARK["teal"])
    ax.axvline(DK_PIVOT, color=DARK["amber"], lw=1, ls="--", label=f"pivot={DK_PIVOT}")
    ax.set_xlabel("dk_estimated (raw signal)")
    ax.set_ylabel("dk_applied  /  VaR bump %")
    ax.set_title("Continuous dk correction\n"
                 "OLD: binary gate at threshold  |  NEW: smooth sigmoid",
                 fontsize=10, fontweight="bold")
    ax.legend(fontsize=8); ax.grid(True, alpha=0.3)
    ax.set_xlim(0, 1)

    fig.suptitle("Continuous dk Correction Design — v5 FINAL",
                 fontsize=12, fontweight="bold")
    fig.tight_layout()
    save_dark(fig, f"{out_dir}/00_continuous_dk_design.png")


def plot_calibration_overview(all_results, out_dir):
    fig,axes=plt.subplots(2,2,figsize=(16,10))
    models=["Historical","Parametric","GARCH","MF-VaR","MF-VaR-Dk"]
    colors=[MODEL_COLORS[m] for m in models]
    x=np.arange(len(models))
    for col,alpha in enumerate(ALPHAS):
        df=aggregate_results(all_results,alpha)
        a_label=f"{int((1-alpha)*100)}% VaR (alpha={alpha})"
        ax=axes[0,col]
        cal=[df.loc[df["Model"]==m,"calibration_err_mean"].values[0] for m in models]
        bars=ax.bar(x,cal,color=colors,alpha=0.85,width=0.6)
        ax.set_xticks(x); ax.set_xticklabels(models,rotation=20,ha="right",fontsize=9)
        ax.set_title(f"Calibration Error |breach_rate - alpha|\n{a_label}",
                     fontsize=10,fontweight="bold")
        ax.set_ylabel("|breach rate - alpha|")
        ax.axhline(0,color=DARK["muted"],lw=0.8,ls="--")
        for bar,v in zip(bars,cal):
            if np.isfinite(v):
                ax.text(bar.get_x()+bar.get_width()/2,bar.get_height()+0.0005,
                        f"{v:.4f}",ha="center",va="bottom",fontsize=8,color=DARK["text"])
        ax.grid(True,alpha=0.3,axis="y")
        ax=axes[1,col]
        kpr=[df.loc[df["Model"]==m,"kupiec_pass_rate"].values[0] for m in models]
        bars=ax.bar(x,kpr,color=colors,alpha=0.85,width=0.6)
        ax.set_xticks(x); ax.set_xticklabels(models,rotation=20,ha="right",fontsize=9)
        ax.set_title(f"Kupiec Pass Rate (p>0.05)\n{a_label}",
                     fontsize=10,fontweight="bold")
        ax.set_ylabel("Fraction passing"); ax.axhline(0.5,color=DARK["muted"],lw=1,ls="--")
        ax.set_ylim(0,1.1)
        for bar,v in zip(bars,kpr):
            if np.isfinite(v):
                ax.text(bar.get_x()+bar.get_width()/2,bar.get_height()+0.01,
                        f"{v:.2f}",ha="center",va="bottom",fontsize=8,color=DARK["text"])
        ax.grid(True,alpha=0.3,axis="y")
    fig.suptitle("VaR Calibration v5-FINAL — MF vs Traditional\nLeft=99% | Right=95%",
                 fontsize=13,fontweight="bold")
    fig.tight_layout()
    save_dark(fig,f"{out_dir}/01_calibration_overview.png")


def plot_es_and_clustering(all_results, out_dir):
    fig,axes=plt.subplots(1,2,figsize=(14,6))
    models=["Historical","Parametric","GARCH","MF-VaR","MF-VaR-Dk"]
    colors=[MODEL_COLORS[m] for m in models]
    x=np.arange(len(models)); alpha=0.05
    df=aggregate_results(all_results,alpha)
    ax=axes[0]
    esr=[df.loc[df["Model"]==m,"es_ratio_mean"].values[0] for m in models]
    bars=ax.bar(x,esr,color=colors,alpha=0.85,width=0.6)
    ax.axhline(1.0,color=DARK["amber"],lw=2,ls="--",label="Ideal=1.0")
    ax.set_xticks(x); ax.set_xticklabels(models,rotation=20,ha="right",fontsize=9)
    ax.set_title("Expected Shortfall Ratio (realised/model)\n<1 over-estimates | >1 under-estimates",
                 fontsize=10,fontweight="bold")
    ax.set_ylabel("ES ratio"); ax.legend(fontsize=9)
    for bar,v in zip(bars,esr):
        if np.isfinite(v):
            ax.text(bar.get_x()+bar.get_width()/2,bar.get_height()+0.01,
                    f"{v:.3f}",ha="center",va="bottom",fontsize=8,color=DARK["text"])
    ax.grid(True,alpha=0.3,axis="y")
    ax=axes[1]
    data={m:[] for m in models}
    for r in all_results:
        if r is None: continue
        a_key=f"a{int(alpha*100):02d}"
        if a_key not in r: continue
        for m in models:
            if m in r[a_key]:
                v=r[a_key][m].get("breach_autocorr")
                if v is not None and np.isfinite(v): data[m].append(v)
    bp=ax.boxplot([data[m] for m in models],patch_artist=True,widths=0.5,
                  medianprops=dict(color=DARK["amber"],lw=2),
                  whiskerprops=dict(color=DARK["muted"]),
                  capprops=dict(color=DARK["muted"]),
                  flierprops=dict(marker="o",ms=3,alpha=0.4,markeredgewidth=0))
    for bx,col in zip(bp["boxes"],colors): bx.set_facecolor(col); bx.set_alpha(0.7)
    for xi,(m,col) in enumerate(zip(models,colors),1):
        jitter=np.random.uniform(-0.15,0.15,len(data[m]))
        ax.scatter(np.full(len(data[m]),xi)+jitter,data[m],color=col,s=20,alpha=0.6,zorder=3)
        if data[m]:
            ax.text(xi,np.mean(data[m]),f"mu={np.mean(data[m]):.3f}",
                    ha="center",va="bottom",fontsize=8,color=DARK["amber"],fontweight="bold")
    ax.axhline(0,color=DARK["muted"],lw=1,ls="--",label="IID ideal")
    ax.set_xticks(range(1,len(models)+1)); ax.set_xticklabels(models,fontsize=9)
    ax.set_ylabel("Lag-1 breach autocorrelation (lower=better)")
    ax.set_title("Breach Clustering — 95% VaR\nLower = breaches more IID",
                 fontsize=10,fontweight="bold")
    ax.legend(fontsize=8); ax.grid(True,alpha=0.3,axis="y")
    fig.suptitle("ES Ratio & Breach Clustering — v5-FINAL",
                 fontsize=12,fontweight="bold")
    fig.tight_layout()
    save_dark(fig,f"{out_dir}/02_es_and_clustering.png")


def plot_dk_distribution(all_results, out_dir):
    dk_vals=[(r["ticker"], r["dk_estimated"], r["dk_applied"], r["dk_n_flagged"])
             for r in all_results if r is not None]
    dk_vals.sort(key=lambda x:-x[1])
    tickers  = [x[0] for x in dk_vals]
    est_vals = [x[1] for x in dk_vals]
    app_vals = [x[2] for x in dk_vals]

    sec_colors={
        "Tech":DARK["blue"],"Financials":DARK["green"],
        "Healthcare & Consumer":DARK["amber"],"Energy & Industrials":DARK["red"],
        "Commodities & Crypto":DARK["purple"],"Intl ETFs":DARK["teal"],
    }
    colors=[sec_colors.get(SECTOR_OF.get(t,"Tech"),DARK["blue"]) for t in tickers]

    fig,axes=plt.subplots(1,3,figsize=(22,6))

    ax=axes[0]
    xi=np.arange(len(tickers))
    ax.bar(xi, est_vals, color=colors, alpha=0.6, width=0.7, label="dk_estimated")
    ax.bar(xi, app_vals, color=colors, alpha=1.0, width=0.4, label="dk_applied (sigmoid)")
    ax.axhline(DK_PIVOT, color=DARK["amber"], lw=1.5, ls="--",
               label=f"DK_PIVOT={DK_PIVOT}")
    ax.set_xticks(xi); ax.set_xticklabels(tickers,rotation=90,fontsize=7)
    ax.set_ylabel("dk value")
    ax.set_title("dk_estimated vs dk_applied (continuous sigmoid)\n"
                 "Wide bar=estimated | Narrow bar=applied",
                 fontsize=10,fontweight="bold")
    for sec,col in sec_colors.items():
        ax.scatter([],[],color=col,s=40,label=sec)
    ax.legend(fontsize=7,ncol=2); ax.grid(True,alpha=0.3,axis="y")

    ax=axes[1]
    ax.hist(est_vals, bins=20, color=DARK["blue"], alpha=0.7,
            edgecolor=DARK["spine"], label="dk_estimated")
    ax.hist(app_vals, bins=20, color=DARK["purple"], alpha=0.7,
            edgecolor=DARK["spine"], label="dk_applied")
    ax.axvline(np.mean(est_vals),color=DARK["amber"],lw=2,ls="--",
               label=f"mean_est={np.mean(est_vals):.3f}")
    ax.axvline(DK_PIVOT,color=DARK["green"],lw=1.5,ls=":",
               label=f"pivot={DK_PIVOT}")
    ax.set_xlabel("dk value"); ax.set_ylabel("Count")
    ax.set_title("Distribution: estimated vs applied\n(sigmoid compresses large values)",
                 fontsize=10,fontweight="bold")
    ax.legend(fontsize=9); ax.grid(True,alpha=0.3,axis="y")

    ax=axes[2]
    # Show VaR multiplier per ticker per alpha
    for alpha, col in zip(ALPHAS, [DARK["purple"], DARK["teal"]]):
        eta  = ETA_BY_ALPHA[alpha]
        mults = [1.0 + eta * v for v in app_vals]
        ax.plot(range(len(tickers)), mults, color=col, lw=1.5,
                marker="o", ms=4, label=f"alpha={alpha} ETA={eta}")
    ax.set_xticks(range(len(tickers)))
    ax.set_xticklabels(tickers, rotation=90, fontsize=7)
    ax.axhline(1.0, color=DARK["muted"], lw=0.8, ls="--", label="no correction")
    ax.set_ylabel("VaR multiplier (1 + ETA × dk_applied)")
    ax.set_title("Continuous dk VaR multiplier per ticker\n"
                 "Shows proportional, non-binary correction",
                 fontsize=10,fontweight="bold")
    ax.legend(fontsize=9); ax.grid(True,alpha=0.3,axis="y")

    fig.suptitle("s_k Non-Constancy — Continuous dk Design — v5 FINAL",
                 fontsize=12,fontweight="bold")
    fig.tight_layout()
    save_dark(fig,f"{out_dir}/03_dk_distribution.png")


def plot_summary_table(all_results, out_dir):
    alpha=0.05; a_key=f"a{int(alpha*100):02d}"
    models=["Historical","Parametric","GARCH","MF-VaR","MF-VaR-Dk"]
    sector_data={sec:{m:{"cal":[],"kp":[]} for m in models} for sec in SECTORS}
    for r in all_results:
        if r is None or a_key not in r: continue
        sec=SECTOR_OF.get(r["ticker"],"Tech")
        for m in models:
            if m not in r[a_key]: continue
            ev=r[a_key][m]
            cal=ev.get("calibration_err"); kp=ev.get("kupiec_pass")
            if cal is not None and np.isfinite(cal): sector_data[sec][m]["cal"].append(cal)
            if kp is not None: sector_data[sec][m]["kp"].append(float(kp))
    rows=[]
    for sec in SECTORS:
        row=[sec]
        for m in models:
            cal_v=sector_data[sec][m]["cal"]; kp_v=sector_data[sec][m]["kp"]
            row.append(f"{np.mean(cal_v):.4f}" if cal_v else "nan")
            row.append(f"{np.mean(kp_v):.2f}"  if kp_v  else "nan")
        rows.append(row)
    cols=["Sector"]
    for m in models:
        sh=m.replace("Historical","Hist").replace("Parametric","Param")
        cols+=[f"{sh} CalErr",f"{sh} KupPass"]
    fig,ax=plt.subplots(figsize=(26,5)); ax.axis("off")
    tbl=ax.table(cellText=rows,colLabels=cols,loc="center",cellLoc="center")
    tbl.auto_set_font_size(False); tbl.set_fontsize(7.5); tbl.scale(1.0,2.2)
    for j in range(len(cols)):
        tbl[0,j].set_facecolor("#1f2937")
        tbl[0,j].set_text_props(color=DARK["blue"],fontweight="bold")
    for i,row in enumerate(rows):
        try:
            cal_floats=[float(row[j]) for j in range(1,len(row),2) if row[j]!="nan"]
            best_cal=min(cal_floats) if cal_floats else None
        except ValueError:
            best_cal=None
        for j in range(len(row)):
            c=tbl[i+1,j]
            c.set_facecolor(DARK["ax"]); c.set_text_props(color=DARK["text"])
            if j>0 and j%2==1 and best_cal is not None:
                try:
                    if abs(float(row[j])-best_cal)<1e-6:
                        c.set_facecolor("#0a2e1a")
                        c.set_text_props(color=DARK["green"],fontweight="bold")
                except ValueError:
                    pass
    ax.set_title(
        "Sector Summary v5-FINAL — 95% VaR | CalErr=|breach_rate-0.05| | "
        "KupPass=fraction passing Kupiec p>0.05 | Green=best in sector",
        fontsize=9,fontweight="bold",color=DARK["text"],pad=15)
    fig.tight_layout()
    save_dark(fig,f"{out_dir}/04_summary_table.png")


def plot_interpretation_heatmap(all_results, out_dir):
    """Heatmap: classification of each ticker — non-Gaussianity vs misspecification."""
    labels = []
    dk_est = []
    esr_01 = []
    bac_01 = []
    kp_01  = []

    for r in all_results:
        if r is None: continue
        ev = r.get("eval_by_alpha", {}).get(0.01, {})
        mf = ev.get("MF-VaR", {})
        labels.append(r["ticker"])
        dk_est.append(r["dk_estimated"])
        esr_01.append(mf.get("es_ratio", np.nan))
        bac_01.append(mf.get("breach_autocorr", np.nan))
        kp_01.append(1.0 if mf.get("kupiec_pass") else 0.0)

    dk_est = np.array(dk_est); esr_01=np.array(esr_01)
    bac_01 = np.array(bac_01); kp_01 =np.array(kp_01)

    fig, axes = plt.subplots(1, 2, figsize=(18, 7))

    # Scatter: dk_estimated vs |ESR-1| — coloured by kupiec_pass
    ax = axes[0]
    esr_dev = np.abs(esr_01 - 1.0)
    pass_mask = kp_01 == 1.0
    ax.scatter(dk_est[pass_mask],  esr_dev[pass_mask],
               color=DARK["green"], s=60, zorder=3, label="MF-VaR Kupiec PASS", alpha=0.85)
    ax.scatter(dk_est[~pass_mask], esr_dev[~pass_mask],
               color=DARK["red"],   s=60, zorder=3, label="MF-VaR Kupiec FAIL", alpha=0.85,
               marker="x", linewidths=2)
    for i, lbl in enumerate(labels):
        ax.annotate(lbl, (dk_est[i], esr_dev[i]), fontsize=6,
                    color=DARK["muted"], xytext=(3, 2), textcoords="offset points")
    ax.axvline(DK_PIVOT, color=DARK["amber"], lw=1, ls="--", label=f"dk pivot={DK_PIVOT}")
    ax.axhline(0.20,     color=DARK["blue"],  lw=1, ls=":",  label="|ESR-1|=0.20")
    ax.set_xlabel("dk_estimated (cumulant non-constancy)")
    ax.set_ylabel("|ES ratio − 1|  (severity misspecification)")
    ax.set_title("Structural non-Gaussianity vs severity misspecification\n"
                 "Top-right = both; Bottom-left = neither",
                 fontsize=10, fontweight="bold")
    ax.legend(fontsize=8); ax.grid(True, alpha=0.3)
    # Quadrant labels
    xlim=ax.get_xlim(); ylim=ax.get_ylim()
    ax.text(xlim[0]+0.002, ylim[1]*0.9, "low dk,\nhigh severity\n→ vol misspec",
            fontsize=7, color=DARK["blue"],  alpha=0.7)
    ax.text(xlim[1]*0.6,   ylim[1]*0.9, "high dk,\nhigh severity\n→ full misspec",
            fontsize=7, color=DARK["red"],   alpha=0.7)
    ax.text(xlim[0]+0.002, ylim[0]+0.01, "low dk,\nlow severity\n→ well-specified",
            fontsize=7, color=DARK["green"], alpha=0.7)
    ax.text(xlim[1]*0.6,   ylim[0]+0.01, "high dk,\nlow severity\n→ struct. non-Gauss",
            fontsize=7, color=DARK["amber"], alpha=0.7)

    # Scatter: dk_estimated vs breach_AC
    ax = axes[1]
    ax.scatter(dk_est[pass_mask],  bac_01[pass_mask],
               color=DARK["green"], s=60, zorder=3, label="MF-VaR Kupiec PASS", alpha=0.85)
    ax.scatter(dk_est[~pass_mask], bac_01[~pass_mask],
               color=DARK["red"],   s=60, zorder=3, label="MF-VaR Kupiec FAIL", alpha=0.85,
               marker="x", linewidths=2)
    for i, lbl in enumerate(labels):
        ax.annotate(lbl, (dk_est[i], bac_01[i]), fontsize=6,
                    color=DARK["muted"], xytext=(3, 2), textcoords="offset points")
    ax.axvline(DK_PIVOT, color=DARK["amber"], lw=1, ls="--", label=f"dk pivot={DK_PIVOT}")
    ax.axhline(0.10, color=DARK["red"],  lw=1, ls=":", label="AC=0.10 (clustering)")
    ax.axhline(0.0,  color=DARK["muted"],lw=0.8, ls="--", label="IID ideal")
    ax.set_xlabel("dk_estimated (cumulant non-constancy)")
    ax.set_ylabel("Breach autocorrelation (lag-1)")
    ax.set_title("dk signal vs breach clustering\n"
                 "High AC + high dk → regime-driven misspecification",
                 fontsize=10, fontweight="bold")
    ax.legend(fontsize=8); ax.grid(True, alpha=0.3)

    fig.suptitle("Interpretation: structural non-Gaussianity vs model misspecification — v5-FINAL",
                 fontsize=12, fontweight="bold")
    fig.tight_layout()
    save_dark(fig, f"{out_dir}/05_interpretation_heatmap.png")


# ═══════════════════════════════════════════════════════════════════════════════
#  MAIN
# ═══════════════════════════════════════════════════════════════════════════════
def main():
    t_total=time.time()
    print("="*70)
    print("  MF Risk Benchmark v5 FINAL")
    print()
    print("  KEY CHANGES:")
    print("  1. CONTINUOUS dk correction (sigmoid) replaces binary gate")
    print("     dk_applied = dk_estimated × σ((dk_est − pivot) / scale)")
    print("     Applied at ALL alphas; ETA scaled per alpha level")
    print()
    print("  2. FULL 7-SECTION DIAGNOSTIC per ticker:")
    print("     1. Non-constancy  2. D_k table  3. Significance")
    print("     4. VaR context    5. Activation  6. Counterfactual")
    print("     7. Interpretation (non-Gaussianity vs misspecification)")
    print()
    print("  3. dk_estimated AND dk_applied always printed — never suppressed")
    print()
    print(f"  Sigmoid: pivot={DK_PIVOT}  scale={DK_SCALE}")
    print(f"  ETA:     alpha=0.01 → {ETA_BY_ALPHA[0.01]}  |  alpha=0.05 → {ETA_BY_ALPHA[0.05]}")
    print(f"  50 assets × 6 sectors × 2010-2026 × alpha={ALPHAS}")
    print("="*70)

    all_data=batch_download()

    # Plot the sigmoid design before any ticker runs
    plot_continuous_dk_curve(OUT_DIR)

    print("\nRunning GPU Optuna calibration...")
    best_params=calibrate_mf_gpu(all_data)
    MF_PARAMS[0.01]=best_params[0.01]
    MF_PARAMS[0.05]=best_params[0.05]

    print("\nFinal MF_PARAMS (v5-FINAL calibrated):")
    for a,p in MF_PARAMS.items():
        print(f"  alpha={a}: gamma={p['gamma']:.4f}  beta={p['beta']:.4f}  "
              f"delta={p['delta']:.4f}")

    all_results=[]
    for sector,tickers in SECTORS.items():
        print(f"\n{'─'*70}")
        print(f"  SECTOR: {sector}")
        print(f"{'─'*70}")
        for ticker in tickers:
            t0=time.time()
            print(f"\n  Processing {ticker}...")
            result=benchmark_ticker(
                ticker,
                all_data.get(ticker),
                verbose_diag=True,   # ← full 7-section diagnostic per ticker
            )
            if result is None:
                print(f"  {ticker}: SKIPPED (insufficient data)")
                all_results.append(None)
                continue
            all_results.append(result)
            print(f"  [{ticker}] wall time: {time.time()-t0:.0f}s  "
                  f"dk_est={result['dk_estimated']:.4f}  "
                  f"dk_app={result['dk_applied']:.4f}")

    print("\n\nGenerating plots...")
    plot_continuous_dk_curve(OUT_DIR)
    plot_calibration_overview(all_results, OUT_DIR)
    plot_es_and_clustering(all_results, OUT_DIR)
    plot_dk_distribution(all_results, OUT_DIR)
    plot_summary_table(all_results, OUT_DIR)
    plot_interpretation_heatmap(all_results, OUT_DIR)

    # ── OVERALL RESULTS TABLE ──
    print("\n"+"="*70)
    print("  OVERALL RESULTS  (v5 FINAL — continuous dk)")
    print("="*70)
    hdr=(f"  {'Model':<14} {'CalErr':>8} {'KupPass':>8} {'CC_Pass':>8}"
         f" {'ES_ratio':>9} {'CapEff':>10} {'Breach_AC':>10}")
    for alpha in ALPHAS:
        df=aggregate_results(all_results,alpha)
        print(f"\n  alpha={alpha} ({int((1-alpha)*100)}% VaR):")
        print(hdr); print("  "+"-"*80)
        for _,row in df.iterrows():
            print(f"  {row['Model']:<14}"
                  f" {row.get('calibration_err_mean',np.nan):>8.4f}"
                  f" {row.get('kupiec_pass_rate',np.nan):>8.3f}"
                  f" {row.get('cc_pass_rate',np.nan):>8.3f}"
                  f" {row.get('es_ratio_mean',np.nan):>9.3f}"
                  f" {row.get('capital_eff_mean',np.nan):>10.5f}"
                  f" {row.get('breach_autocorr_mean',np.nan):>10.4f}")

    # ── dk SIGNAL SUMMARY ──
    all_est  = [r["dk_estimated"] for r in all_results if r is not None]
    all_app  = [r["dk_applied"]   for r in all_results if r is not None]
    print(f"\n  dk signal summary (continuous design):")
    print(f"    dk_estimated: mean={np.mean(all_est):.4f}  "
          f"median={np.median(all_est):.4f}  max={np.max(all_est):.4f}")
    print(f"    dk_applied:   mean={np.mean(all_app):.4f}  "
          f"median={np.median(all_app):.4f}  max={np.max(all_app):.4f}")
    print(f"    sigmoid compression ratio: "
          f"{np.mean(all_app)/max(np.mean(all_est),1e-9):.3f}  "
          f"(1.0 = no compression)")
    above_pivot = sum(e > DK_PIVOT for e in all_est)
    print(f"    dk_est above sigmoid pivot ({DK_PIVOT}): "
          f"{above_pivot}/{len(all_est)}  (paper reports 10/50)")

    for alpha in ALPHAS:
        eta = ETA_BY_ALPHA[alpha]
        mults = [1.0 + eta * v for v in all_app]
        print(f"    alpha={alpha} ETA={eta}  VaR multiplier range: "
              f"[{min(mults):.4f}, {max(mults):.4f}]  "
              f"mean={np.mean(mults):.4f}")

    print(f"\n  Wall time: {time.time()-t_total:.1f}s")
    print(f"  Outputs → {OUT_DIR}/")
    print(f"    00_continuous_dk_design.png")
    print(f"    01_calibration_overview.png")
    print(f"    02_es_and_clustering.png")
    print(f"    03_dk_distribution.png")
    print(f"    04_summary_table.png")
    print(f"    05_interpretation_heatmap.png")
    print("="*70)


if __name__ == "__main__":
    main()

[GPU] CuPy detected — 8.0 GB total, 6.9 GB free
  MF Risk Benchmark v5 FINAL

  KEY CHANGES:
  1. CONTINUOUS dk correction (sigmoid) replaces binary gate
     dk_applied = dk_estimated × σ((dk_est − pivot) / scale)
     Applied at ALL alphas; ETA scaled per alpha level

  2. FULL 7-SECTION DIAGNOSTIC per ticker:
     1. Non-constancy  2. D_k table  3. Significance
     4. VaR context    5. Activation  6. Counterfactual
     7. Interpretation (non-Gaussianity vs misspecification)

  3. dk_estimated AND dk_applied always printed — never suppressed

  Sigmoid: pivot=0.1  scale=0.05
  ETA:     alpha=0.01 → 0.25  |  alpha=0.05 → 0.12
  50 assets × 6 sectors × 2010-2026 × alpha=[0.01, 0.05]
  Got 50/50 tickers
  Saved: mf_risk_outputs_v5_final/00_continuous_dk_design.png

Running GPU Optuna calibration...

  v5-FINAL GPU Calibration  (GPU=YES)
  Tickers: 20  Trials/stage: 150  Batch: 8

[Precompute] Extracting features for 20 tickers...


KeyboardInterrupt: 